In [1]:
!pip install pandas pyarrow datasets transformers torch # Or tensorflow

Imports

In [2]:
# Install if you haven't already
# !pip install datasets transformers torch sklearn seqeval accelerate -q # Use tensorflow if you prefer

import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split # To create a validation set

Load with Pandas:

In [3]:
try:
    file_path = 'train.parquet' # Adjust path if needed
    df = pd.read_parquet(file_path)

    print("Successfully loaded train.parquet into pandas DataFrame.")
    print("DataFrame Info:")
    df.info()
    print("\nFirst 5 rows:")
    print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

    # Verify the 'trigger_words' column type and content for the first manipulative row
    first_manipulative_row = df[df['manipulated'] == True].iloc[0]
    print(f"\nType of 'trigger_words' column: {type(first_manipulative_row['trigger_words'])}")
    print(f"Content of 'trigger_words' for first manipulative example (id: {first_manipulative_row['id']}):")
    print(first_manipulative_row['trigger_words'])

except Exception as e:
    print(f"Error loading Parquet file: {e}")
    print("Please ensure 'train.parquet' is uploaded correctly and the path is right.")

Successfully loaded train.parquet into pandas DataFrame.
DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3822 entries, 0 to 3821
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             3822 non-null   object
 1   content        3822 non-null   object
 2   lang           3822 non-null   object
 3   manipulative   3822 non-null   bool  
 4   techniques     2589 non-null   object
 5   trigger_words  2589 non-null   object
dtypes: bool(1), object(5)
memory usage: 153.2+ KB

First 5 rows:
| id                                   | content                                                                                                                                                                                                                                                                                                                      | lang   | manipulative   | techniques               

Hugging Face Dataset

In [4]:
# Convert pandas DataFrame to Hugging Face Dataset
full_dataset = Dataset.from_pandas(df)

# Split the dataset into training and validation sets (e.g., 90% train, 10% validation)
# Use a fixed seed for reproducibility
train_val_split = full_dataset.train_test_split(test_size=0.1, seed=42)

# Rename for clarity
train_dataset = train_val_split['train']
eval_dataset = train_val_split['test'] # 'test' is the default name for the validation split

print("Converted DataFrame to Dataset and split into train/eval sets:")
print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(eval_dataset)}")
print("\nExample from train_dataset:")
print(train_dataset[0]) # Show one example

Converted DataFrame to Dataset and split into train/eval sets:
Training examples: 3439
Validation examples: 383

Example from train_dataset:
{'id': '6e9be399-dd7e-4e80-a320-361fa540e27a', 'content': 'Ми наводимо – ви нищите. День у день, аж до перемоги. Працювати на передовій пліч-о-пліч із богами війни, чітко виконувати завдання у тісній зв’язці з кращими – то велика честь!  \nУ День ракетних військ і артилерії пригадуємо найзапальніші моменти спільної роботи зі знищення ворога. \nЗ вашим днем, побратими!\n💪', 'lang': 'uk', 'manipulative': True, 'techniques': ['loaded_language', 'euphoria', 'glittering_generalities'], 'trigger_words': [[0, 177]]}


Tokenizer

In [5]:
# Using xlm-roberta-base as a strong multilingual starting point
model_checkpoint = "xlm-roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print(f"Loaded tokenizer: {model_checkpoint}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded tokenizer: xlm-roberta-large


Tokenization and Alignment Function

In [6]:
def tokenize_and_align_labels(example):
    # Tokenize the content, returning offsets to map tokens back to original characters
    tokenized_inputs = tokenizer(example["content"],
                                 truncation=True,          # Truncate long sequences
                                 max_length=512,           # Standard max length for many BERT-like models
                                 return_offsets_mapping=True) # << Needed for alignment

    # Get the list of [start_char, end_char] manipulative spans
    char_spans = example["trigger_words"]

    # If char_spans is None (missing), treat it as an empty list
    if char_spans is None:
        char_spans = []

    # Initialize labels list with -100 (ignored by loss function)
    # We'll overwrite with 0 or 1 for actual content tokens.
    labels = np.full(len(tokenized_inputs["input_ids"]), -100, dtype=int)

    # Get the character offset mapping for each token
    offset_mapping = tokenized_inputs["offset_mapping"]

    # Iterate through each token and its character offsets
    for token_idx, (token_char_start, token_char_end) in enumerate(offset_mapping):
        # If offset is (0, 0), it's a special token ([CLS], [SEP], [PAD]), leave label as -100
        if token_char_start == 0 and token_char_end == 0:
            continue

        # Default label for content tokens is 0 (non-manipulative)
        current_label = 0

        # Check if this token's character range overlaps with ANY manipulative span
        for span_char_start, span_char_end in char_spans:
            # Check for overlap (assuming span_end is exclusive)
            # Does token start within the span?
            cond1 = token_char_start >= span_char_start and token_char_start < span_char_end
            # Does token end within the span?
            cond2 = token_char_end > span_char_start and token_char_end <= span_char_end
            # Does token completely contain the span?
            cond3 = token_char_start <= span_char_start and token_char_end >= span_char_end

            if cond1 or cond2 or cond3:
                current_label = 1 # Mark as manipulative
                break # No need to check other spans for this token

        labels[token_idx] = current_label

    tokenized_inputs["labels"] = labels.tolist()
    # Can remove offset mapping now if not needed later
    # tokenized_inputs.pop("offset_mapping")
    return tokenized_inputs

print("Defined the tokenize_and_align_labels function.")

Defined the tokenize_and_align_labels function.


Apply Function To Dataset

In [7]:
print("Applying tokenization and alignment to train and validation datasets...")

# We use remove_columns to keep only the columns the model expects
# Adjust columns_to_remove based on the exact columns in your dataset after splitting
columns_to_remove = ['id', 'content', 'lang', 'manipulative', 'techniques', 'trigger_words']
# Make sure not to remove essential columns if you added any others

tokenized_train_dataset = train_dataset.map(
    tokenize_and_align_labels,
    batched=False, # Process example by example for simplicity
    remove_columns=columns_to_remove
)

tokenized_eval_dataset = eval_dataset.map(
    tokenize_and_align_labels,
    batched=False,
    remove_columns=columns_to_remove
)

print("Processing complete.")
print("\nTokenized Train Dataset example:")
print(tokenized_train_dataset[0]) # Show structure of the first processed example

Applying tokenization and alignment to train and validation datasets...


Map:   0%|          | 0/3439 [00:00<?, ? examples/s]

Map:   0%|          | 0/383 [00:00<?, ? examples/s]

Processing complete.

Tokenized Train Dataset example:
{'input_ids': [0, 5209, 83176, 1455, 46, 1045, 1688, 21773, 5, 51208, 84, 6365, 4, 19318, 255, 93565, 89, 5, 8498, 1976, 40170, 29, 6962, 122758, 3707, 37792, 9, 197, 9, 1078, 37792, 4459, 73107, 1281, 72549, 4, 199045, 180835, 52432, 84, 168902, 7811, 11781, 26, 7736, 2475, 210, 7774, 117984, 46, 690, 53691, 135953, 38, 447, 51208, 61295, 1315, 55810, 189, 22793, 52886, 79792, 440, 22302, 68610, 2032, 1323, 59933, 194777, 102967, 72521, 2401, 11581, 9234, 136025, 13191, 213483, 59, 5, 1522, 142736, 132709, 4, 129, 55428, 827, 38, 6, 238992, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'offset_mapping': [[0, 0], [0, 2], [3, 9], [9, 11], [12, 13], [14, 16], [17, 19], [19, 23], [23, 24]

Model Selection

In [8]:
# Make sure necessary libraries are imported
from transformers import AutoModelForTokenClassification, AutoConfig
# import torch # Or tensorflow

# Ensure model_checkpoint matches the tokenizer used in Step 2
model_checkpoint = "xlm-roberta-large"

# Define label mappings
id2label = {0: "O", 1: "MANIP"} # O=Outside, MANIP=Manipulative span
label2id = {"O": 0, "MANIP": 1}

# Load model configuration and specify number of labels
config = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

# Load the pre-trained model with a token classification head
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    config=config
)

print(f"Loaded model architecture: {model_checkpoint} for Token Classification")

# Send model to GPU (if using PyTorch and not relying solely on Trainer)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# print(f"Model moved to device: {device}")

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded model architecture: xlm-roberta-large for Token Classification


Compute Metrics Function

In [9]:
# Import individual score functions and accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2) # Get label ids from logits

    # --- Important: Remove ignored index (-100) ---
    # Flatten predictions and labels, ignoring -100
    true_predictions = [
        p for pred, lab in zip(predictions, labels) for (p, l) in zip(pred, lab) if l != -100
    ]
    true_labels = [
        l for pred, lab in zip(predictions, labels) for (p, l) in zip(pred, lab) if l != -100
    ]

    # --- Calculate metrics individually for the positive class (label 1) ---
    precision = precision_score(
        y_true=true_labels,
        y_pred=true_predictions,
        labels=[1],          # Specify the positive class label
        average='binary',    # Calculate for the specified positive label
        zero_division=0
    )
    recall = recall_score(
        y_true=true_labels,
        y_pred=true_predictions,
        labels=[1],
        average='binary',
        zero_division=0
    )
    f1 = f1_score(
        y_true=true_labels,
        y_pred=true_predictions,
        labels=[1],
        average='binary',
        zero_division=0
    )
    # --- End of individual calculation ---

    # Calculate overall accuracy just for reference
    accuracy = accuracy_score(true_labels, true_predictions)

    return {
        'accuracy': accuracy,
        'f1_positive': f1, # F1 score for the 'manipulative' class (label 1)
        'precision_positive': precision,
        'recall_positive': recall
    }

print("Defined compute_metrics function using individual score functions.")

Defined compute_metrics function using individual score functions.


Training Arguments

In [10]:
from transformers import TrainingArguments

# Adjust batch size based on T4 memory (8 or 16 often work for 'base' models)
batch_size = 4 # Let's try 16 for xlm-r-base on T4

args = TrainingArguments(
    output_dir=f"{model_checkpoint}-finetuned-unlp-span", # Directory to save model checkpoints
    eval_strategy="steps",          # Evaluate during training
    eval_steps=100,                       # Evaluate every 100 steps (adjust frequency as needed)
    logging_steps=100,                    # Log metrics every 100 steps
    save_strategy="steps",                # Save checkpoints based on evaluation steps
    save_steps=100,                       # Save checkpoint every 100 steps
    learning_rate=2e-5,                   # Good default for fine-tuning
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size * 2, # Can often use larger eval batch size
    num_train_epochs=5,                   # Start with 5, early stopping will quit sooner if needed
    weight_decay=0.01,                    # Standard weight decay
    load_best_model_at_end=True,          # <<<<< Load best model based on metric
    metric_for_best_model="f1_positive",  # <<<<< Metric to determine the "best" model
    greater_is_better=True,               # <<<<< Maximize F1 score
    save_total_limit=2,                   # Optional: Save only the best and the latest checkpoints
    fp16=True,                            # Use mixed precision on T4 for faster training
    report_to="none",                     # Disable default reporting (like wandb)
)

print("Defined TrainingArguments with early stopping.")

Defined TrainingArguments with early stopping.


Instantiate Training

In [11]:
from transformers import Trainer, DataCollatorForTokenClassification

# Data collator pads sequences to the longest sequence in a batch
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train_dataset,  # Processed training data from Step 2
    eval_dataset=tokenized_eval_dataset,    # Processed validation data from Step 2
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics         # Function to compute metrics during evaluation
)

print("Trainer instantiated.")

<ipython-input-11-a14d52e23f99>:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Trainer instantiated.


Training

In [12]:
# This starts the fine-tuning process based on your configuration
print("Starting model training...")
training_results = trainer.train()
print("Training finished.")

# Optional: Save the final (best) model, tokenizer, and training state
# The Trainer already saved checkpoints during training in the output_dir,
# and loaded the best one into trainer.model if load_best_model_at_end=True.
# This explicitly saves the final state of the best model.
final_model_path = f"./{model_checkpoint}-final"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"Best model saved to {final_model_path}")

# Optional: Look at the training results summary
print("\nTraining Results Summary:")
print(training_results)

Starting model training...


Step,Training Loss,Validation Loss,Accuracy,F1 Positive,Precision Positive,Recall Positive
100,0.520400,0.523633,0.767861,0.044361,0.877493,0.022756
200,0.494800,0.454495,0.771622,0.074310,0.922535,0.038714
300,0.478800,0.446076,0.795046,0.291314,0.803470,0.177909
400,0.501800,0.464627,0.767703,0.057893,0.728571,0.030144
500,0.473700,0.418733,0.799804,0.360027,0.740511,0.237828
600,0.477700,0.384362,0.823350,0.515079,0.735766,0.396232
700,0.488100,0.386512,0.823910,0.551586,0.694603,0.457407
800,0.462300,0.405056,0.821269,0.548100,0.682830,0.457776
900,0.427200,0.409395,0.826044,0.592859,0.664891,0.534909
1000,0.370200,0.387370,0.826762,0.574668,0.686295,0.494274


Training finished.
Best model saved to ./xlm-roberta-large-final

Training Results Summary:
TrainOutput(global_step=4300, training_loss=0.3095521605292032, metrics={'train_runtime': 4547.6433, 'train_samples_per_second': 3.781, 'train_steps_per_second': 0.946, 'total_flos': 9100532229163884.0, 'train_loss': 0.3095521605292032, 'epoch': 5.0})
